In [1]:
# imports
import MDAnalysis as mda
import pandas as pd
import nglview as nv
from tqdm.notebook import tqdm
from setup_structures_cylinder import assign_segment_ids, identify_subaggregates
from merge_structures_cylinder_new import merge_structures

In [ ]:
# load the structure files, assign unique segment IDs for every protein, identify aggregates in the structures
# and save relevant paramenters in a pandas dataframe
existing_segids = {}
protein_data = {}
for prot in tqdm(["hpl2", "lin13", "lin65", "met2"], desc="Loading structure files", leave=False, unit="Files"):
    x = mda.Universe(f"input_structures/single_comp_structures/{prot}.pdb")
    u = x.copy()

    # Add unique segment IDs to differentiate between protein chains
    u = assign_segment_ids(u, prot, existing_segids)
    
    # assign known universe dimensions
    u.dimensions = [600,600,3000,90,90,90]

    # Analyze chains and clusters
    clusters = identify_subaggregates(u)    
    protein_data[prot] = {
        'universe': u,
        "n_atoms": len(u.atoms),
        'subaggregates': clusters,
        'rg_min_cluster': min([cluster["cluster"].radius_of_gyration() for cluster in clusters]),
        'rg_max': max([prot.atoms.radius_of_gyration() for prot in u.segments]),
        "n_proteins": len(u.segments), 
        "box_dimensions": u.dimensions
    }

proteins_df = pd.DataFrame.from_dict(protein_data, orient='index')
proteins_df

Loading structure files:   0%|          | 0/4 [00:00<?, ?Files/s]

AttributeError: 'str' object has no attribute 'radius_of_gyration'

In [ ]:
view = nv.show_mdanalysis(proteins_df.loc["lin13", "universe"].atoms)

# add the unit cell (the box)
view.add_unitcell()

view

c:\Users\phili\anaconda3\envs\MD_b_practical\Lib\site-packages\MDAnalysis\coordinates\PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


NGLWidget()

In [ ]:
merged = merge_structures(proteins_df)

New Structure file initialized with box dimensions [709.9295739719539, 709.9295739719539, 2000].


Placing structures:   0%|          | 0/4 [00:00<?, ?it/s]

Placed structure lin13 (1 clusters).
Placed structure lin65 (2/2)
Placed structure hpl2 (11/31)
Placed structure met2 (0/31)


Fitting orphans:   0%|          | 0/51 [00:00<?, ?it/s]

Only 0/51 clusters could fit. No clusters placed. 


In [ ]:
view = nv.show_mdanalysis(merged.atoms)

# add the unit cell (the box)
view.add_unitcell()

view

NGLWidget()

In [ ]:
from MDAnalysis.lib.distances import capped_distance

def check_true_structure_overlaps(u, cutoff=10.0):
    """
    Groups segments by their 2-character prefix and checks for 
    collisions between these groups (ignoring internal chain-chain contacts).
    """
    # 1. Group segments by their prefix (first 2 chars)
    # Result: {'H2': AtomGroup, 'L1': AtomGroup, ...}
    prefixes = sorted(list(set(s.segid[:2] for s in u.segments)))
    structure_groups = {p: u.select_atoms(f"segid {p}*") for p in prefixes}
    
    total_collisions = 0
    checked_pairs = []

    # 2. Nested loop to check only DIFFERENT structures
    for i, pref1 in enumerate(prefixes):
        for j, pref2 in enumerate(prefixes):
            if i >= j: continue # Avoid double-counting and self-interaction
            
            group1 = structure_groups[pref1]
            group2 = structure_groups[pref2]
            
            # Use capped_distance for the two distinct AtomGroups
            pairs = capped_distance(group1.positions, 
                                    group2.positions, 
                                    max_cutoff=cutoff, 
                                    box=u.dimensions)
            
            count = len(pairs[0])
            if count > 0:
                print(f"Collision: {pref1} <-> {pref2} | {count} atoms overlapping")
            
            total_collisions += count
            
    return total_collisions

# Run the check
true_overlaps = check_true_structure_overlaps(merged)
print(f"--- Final Report: {true_overlaps} true inter-structure collisions found ---")

--- Final Report: 0 true inter-structure collisions found ---
